# Phase 2 - DocLayNet layout detector smoke (T4)

Runner only (CLAUDE.md P1/P2). The Phase 2 **gate**: confirm `Aryn/deformable-detr-DocLayNet`
loads, exposes `config.id2label` with a Table class, detects a Table box, and that the box
crops - on a real T4 so runtime / VRAM are measured. It pins nothing; pinning
`config.LAYOUT_MODEL` is a manual step after the gate passes.

**Why transformers is pinned to 4.49.0:** the checkpoint was saved with transformers 4.36.2
and uses a timm resnet50 backbone. The v5 meta-init loader does not load that backbone's
weights, so detection collapses to degenerate boxes. Pinning below v5 restores the eager
loader that loads the backbone. Logic lives in `scripts/smoke_layout_detector.py`, not here.

In [ ]:
# Boot - clone-if-absent, pin the Phase 2 dev branch
import os
BRANCH = 'feature/phase2-doclaynet-layout'
if not os.path.isdir('/content/FinDocStructRAG/.git'):
    !git clone --quiet https://github.com/AD2000X/FinDocStructRAG.git /content/FinDocStructRAG
!cd /content/FinDocStructRAG && git fetch origin --quiet && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}
%cd /content/FinDocStructRAG

In [ ]:
# Mount Drive so config.OUTPUT_ROOT (and config.LAYOUT_OUTPUT) resolve to Drive paths
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install - pin transformers below v5 so the eager loader loads the timm backbone.
# torch is preinstalled on Colab; the smoke subprocess picks up this pinned transformers.
!pip install -q "transformers==4.49.0" timm Pillow requests
!python -c "import transformers, torch; print('transformers', transformers.__version__, '| torch', torch.__version__)"

## Step 1 - confirm loading is fixed (example page)

Run the smoke on the model card's example page first. Loading is fixed when: (1) there is **no**
`Materializing param=` line, and (2) the top detections are **large, high-confidence** boxes -
not the top-center ~50px low-score specks that meant the backbone was unloaded. A Table is not
required here (the example page may not have one); this step only checks the model runs correctly.

In [ ]:
!python scripts/smoke_layout_detector.py --threshold 0.3

## Step 2 - the real gate (a page with a Table GT)

Only after Step 1 looks healthy. Pull one DocLayNet val page that has a Table annotation
(`category_id 9 == Table`), then run the smoke on it. The gate passes when a Table box is
detected and cropped (`smoke OK`); the crop is saved so it can be eyeballed.

In [ ]:
# Grab a DocLayNet val page containing a Table (category_id 9 == Table).
!pip install -q datasets
from datasets import load_dataset
from pathlib import Path

out = Path('/content/doclaynet_table_smoke.png')
ds = load_dataset('docling-project/DocLayNet-v1.1', split='val', streaming=True)
for i, ex in enumerate(ds):
    if 9 in ex['category_id']:
        ex['image'].convert('RGB').save(out)
        print('saved', out, '| idx', i, '| size', ex['image'].size, '| cats', set(ex['category_id']))
        break

In [ ]:
!python scripts/smoke_layout_detector.py --image /content/doclaynet_table_smoke.png --threshold 0.3 --save-crop /content/layout_table_crop.png

## Step 3 - factory integration: build_layout_detector + detect_layout + crop

Exercise the actual factory functions from `src/layout_detector.py` and
`src/table_detection.py`. Two sub-steps:

**3a (normal path)** — both detectors built; `detect_layout` runs primary first;
primary finds a 0.907 Table so fallback does NOT fire; table regions above
`MIN_CROP_SCORE` are cropped with the score-filter contract.

**3b (forced fallback)** — set `min_table_score=0.99` so the primary 0.907 table
doesn't qualify; TATR fallback fires and contributes its table; verifies the
sequential path and source labelling.

In [ ]:
# Step 3a - normal detect_layout path (primary wins; fallback should NOT fire)
import time
from PIL import Image
from src.layout_detector import build_layout_detector
from src.table_detection import build_table_transformer_detector
from src.layout_parsing import detect_layout, TABLE_LABEL
from src.bbox_utils import crop_with_padding
from src import config

img = Image.open('/content/doclaynet_table_smoke.png').convert('RGB')

t0 = time.time()
layout_det = build_layout_detector(config.LAYOUT_MODEL, threshold=0.3)
table_det  = build_table_transformer_detector(config.TATR_DETECTION_MODEL, threshold=0.5)
print(f"[load] both detectors ready in {time.time()-t0:.1f}s")

t0 = time.time()
regions = detect_layout(img, layout_det, table_det, min_table_score=0.5)
print(f"[detect] {len(regions)} regions in {time.time()-t0:.2f}s")
for r in sorted(regions, key=lambda r: -r.score):
    print(f"  {r.label:16s}  {r.score:.3f}  {[round(c) for c in r.box]}  [{r.source}]")

# Crop: score filter is the caller's responsibility (detect_layout returns candidates)
MIN_CROP_SCORE = 0.5
tables = [r for r in regions if r.label == TABLE_LABEL and r.score >= MIN_CROP_SCORE]
print(f"\n[crop] {len(tables)} table(s) >= {MIN_CROP_SCORE}")
for i, r in enumerate(tables):
    crop = crop_with_padding(img, r.box, pad=4)
    path = f'/content/factory_table_crop_{i}.png'
    crop.save(path)
    print(f"  crop {i}: box={[round(c) for c in r.box]}  size={crop.size}  saved {path}")

In [ ]:
# Step 3b - forced fallback path: set min_table_score=0.99 so primary 0.907 doesn't qualify;
# TATR fallback must fire and contribute its own table region
regions_fb = detect_layout(img, layout_det, table_det, min_table_score=0.99, dedup_iou=0.5)
tables_fb = [r for r in regions_fb if r.label == TABLE_LABEL]
sources    = {r.source for r in tables_fb}
print(f"[forced-fallback] {len(tables_fb)} table(s)  sources={sources}")
for r in tables_fb:
    print(f"  {r.label:16s}  {r.score:.3f}  [{r.source}]  box={[round(c) for c in r.box]}")
assert len(tables_fb) >= 1, "fallback must produce at least one table"
assert "table_fallback" in sources or "layout" in sources, "unexpected source"
print("fallback path OK")

## Step 4 - layout batch runner (debug subset, seed=7, n=20)

Run `scripts/run_layout_batch.py` on a fixed 20-page DocLayNet val subset.
Artifacts land on Drive under `config.LAYOUT_OUTPUT` (persistent across sessions):
- `regions/<page_id>.json` — full Region list per page
- `crops/<page_id>_table_<i>.png` — one PNG per table above `--table-threshold`
- `manifest.csv` — counters + `fallback_used` flag per page

Goal here: confirm artifact stream is stable and crop quality looks reasonable
before touching AP/IoU. Inspect the manifest summary and spot-check a few crops.

In [ ]:
!python scripts/run_layout_batch.py --seed 7 --n 20 --require-table-gt --primary-threshold 0.3

In [ ]:
# Manifest preview + spot-check one crop
import pandas as pd
from pathlib import Path
from src import config
from IPython.display import display, Image as IPImage

manifest_path = config.LAYOUT_OUTPUT / "manifest.csv"
df = pd.read_csv(manifest_path)
print(df[["page_id", "status", "gt_tables", "num_regions", "num_tables", "num_cropped", "fallback_used"]].to_string(index=False))
print(f"\nprocessed={df['status'].eq('processed').sum()}  failed={df['status'].eq('failed').sum()}")
print(f"gt_table pages={df['gt_tables'].gt(0).sum()}  total crops={df['num_cropped'].sum()}  fallback pages={df['fallback_used'].sum()}")

# Only show crops from this run's pages (avoids stale artifact confusion)
crops_dir = config.LAYOUT_OUTPUT / "crops"
run_pages = set(df["page_id"])
run_crops = sorted(
    f for f in crops_dir.glob("*.png")
    if f.stem.rsplit("_table_", 1)[0] in run_pages
)
first_crop = run_crops[0] if run_crops else None
if first_crop:
    print(f"\nspot-check: {first_crop.name}")
    display(IPImage(str(first_crop), width=600))
else:
    print("no crops written for this run")

## Step 5 - IoU diagnostic (Q1 / Q2 / Q3)

Re-runs detection on the same 20 pages capturing **pre-fallback** primary results
(so primary IoU is not contaminated by dedup removal) and computes per-page IoU
against DocLayNet GT annotations.

Answers:
- **Q1** — did primary miss the table entirely, or just score < 0.5?
- **Q2** — is fallback IoU actually better than primary on those pages?
- **Q3** — is table_threshold=0.5 too strict? (threshold sensitivity table)

Output: `outputs/layout/diagnostic.csv` on Drive.

In [ ]:
!python scripts/eval_layout_iou.py --seed 7 --n 20 --require-table-gt --primary-threshold 0.3

In [ ]:
# Diagnostic CSV preview (sorted by best_iou_crop desc)
import pandas as pd
from src import config

# --require-table-gt writes diagnostic_pos.csv; fall back to diagnostic.csv
_diag = config.LAYOUT_OUTPUT / "diagnostic_pos.csv"
if not _diag.exists():
    _diag = config.LAYOUT_OUTPUT / "diagnostic.csv"
df = pd.read_csv(_diag)
cols = ["page_id", "gt_tables", "num_crop_tables", "matched_50", "matched_75",
        "primary_max_score", "fallback_used",
        "best_iou_primary", "best_iou_crop"]
print(df[cols].sort_values("best_iou_crop", ascending=False).to_string(index=False))
has_gt = df[df["gt_tables"] > 0]
print(f"\ngt_total={has_gt['gt_tables'].sum()}  crops={has_gt['num_crop_tables'].sum()}"
      f"  matched@0.5={has_gt['matched_50'].sum()}  matched@0.75={has_gt['matched_75'].sum()}"
      f"  mean_crop_iou={has_gt['best_iou_crop'].mean():.3f}")

## Step 5b - False-positive diagnostic (pages with no GT table)

Run the same diagnostic on 20 pages that have **no** GT Table annotation.
Any crop produced here is a false positive. Goal: measure false-positive crop rate
and compare fallback behaviour on table-free pages.

Expected: `gt_tables=0` for all rows, `best_iou_*=0.0` throughout.
Watch: how many pages still get a crop? How often does fallback fire?

In [ ]:
!python scripts/eval_layout_iou.py --seed 7 --n 20 --exclude-table-gt --primary-threshold 0.3

## Step 5c - Retune: confirm threshold=0.30 improvement

Q3 simulation (new fallback rule: fires only when `primary_tables >= 1`) predicts
lowering `table_threshold` from 0.50 → 0.30 reduces fallback pages from 3 → 1
and raises `mean_iou_crop_sim` (0.963 reclaimed for the 2 pages where primary
had a low-score but high-IoU box).

Watch the two key numbers in the output:
- `mean best_iou_crop` at thresh=0.30 vs thresh=0.50
- `Fallback used` count (should drop from 3 to 1)

The 2 reclaimed pages are val_000238 (primary score 0.44, IoU 0.954) and
val_001347 (primary max 0.45, IoU 0.949). val_004383 (score 0.335) stays in
fallback territory at both thresholds.

In [ ]:
!python scripts/eval_layout_iou.py --seed 7 --n 20 --require-table-gt --primary-threshold 0.3 --table-threshold 0.3

## Step 5d - Spot-check val_005241: dedup collapse (primary IoU 0.944 → crop IoU 0.610)

`val_005241` has **2 GT tables** that sit close together. Primary found 2 boxes that
overlap each other above `dedup_iou=0.5`, so dedup kept only the higher-scoring one.
That surviving box aligns with GT table 2 (IoU ~0.61), not GT table 1 (IoU ~0.94).
The box that would have given 0.94 was dropped as a "duplicate."

This cell re-runs the primary detector directly (no dedup) and shows per-box IoU against
each GT so the collapse is visible. Also displays any crops saved by Step 4.

**Run Step 4 first** so the regions JSON and crops exist at `config.LAYOUT_OUTPUT`.

In [ ]:
import json
from PIL import Image
from datasets import load_dataset
from IPython.display import display, Image as IPImage
from src import config
from src.bbox_utils import iou, xywh_to_xyxy
from src.layout_detector import build_layout_detector
from src.layout_parsing import TABLE_LABEL

PAGE_ID = "val_005241"
ORIG_IDX = 5241

# GT boxes
ds_val = load_dataset("docling-project/DocLayNet-v1.1", split="val")
ex = ds_val[ORIG_IDX]
bboxes = ex.get("bboxes", ex.get("bbox", []))
cats = ex.get("category_id", [])
gt_boxes = [xywh_to_xyxy(tuple(b)) for cat, b in zip(cats, bboxes) if cat == 9]
print(f"GT tables ({len(gt_boxes)}):")
for i, b in enumerate(gt_boxes):
    print(f"  GT[{i}]: {[round(c) for c in b]}")

# Primary detector (no dedup)
try:
    layout_det
except NameError:
    layout_det = build_layout_detector(config.LAYOUT_MODEL, threshold=0.3)

img = ex["image"].convert("RGB")
primary_tables = [r for r in layout_det(img) if r.label == TABLE_LABEL]
print(f"\nPrimary-alone table regions ({len(primary_tables)}) — before dedup:")
for i, r in enumerate(primary_tables):
    ious_vs_gt = [round(iou(r.box, g), 4) for g in gt_boxes]
    ious_vs_primary = [round(iou(r.box, r2.box), 4) for j, r2 in enumerate(primary_tables) if j != i]
    print(f"  P[{i}] score={r.score:.4f}  box={[round(c) for c in r.box]}")
    print(f"       IoU/GT={ious_vs_gt}  IoU/otherPrimary={ious_vs_primary}")

# Final regions from batch JSON
regions_path = config.LAYOUT_OUTPUT / "regions" / f"{PAGE_ID}.json"
if regions_path.exists():
    regions = json.loads(regions_path.read_text())
    final_tables = [r for r in regions if r["label"] == "table"]
    print(f"\nFinal table regions from batch JSON ({len(final_tables)}) — after dedup:")
    for r in final_tables:
        box = tuple(r["box"])
        ious_vs_gt = [round(iou(box, g), 4) for g in gt_boxes]
        print(f"  score={r['score']:.4f}  source={r['source']}  box={[round(c) for c in box]}")
        print(f"       IoU/GT={ious_vs_gt}")
else:
    print(f"\n[warn] {regions_path} not found — run Step 4 first")

# Crops
crops_dir = config.LAYOUT_OUTPUT / "crops"
crop_files = sorted(crops_dir.glob(f"{PAGE_ID}_table_*.png"))
print(f"\nCrops saved by Step 4 ({len(crop_files)}):")
for crop_path in crop_files:
    print(f"  {crop_path.name}")
    display(IPImage(str(crop_path), width=600))
if not crop_files:
    print("  none — re-run Step 4 (default thresh=0.3 will now save them)")

## Step 5e - dedup sensitivity: test dedup-iou=0.70

`val_005241`: P[0] score=0.484 IoU=0.610, P[1] score=0.342 IoU=0.944.
Their mutual IoU is ~0.600. At `dedup_iou=0.50`, NMS collapses them (keeps P[0] by score);
at `dedup_iou=0.70`, both survive and both GTs get covered.

Expected effect: `best_iou_crop` for val_005241 rises toward 0.944;
the trade-off is potentially one extra crop (two overlapping boxes for the same region).
Watch `mean best_iou_crop` across all 20 pages — it should improve if val_005241 is
the main outlier and other pages are unaffected by the looser dedup.

In [ ]:
!python scripts/eval_layout_iou.py --seed 7 --n 20 --require-table-gt --primary-threshold 0.3 --dedup-iou 0.7

## Step 6 - MVP evaluation (seed=42, n=200)

Final calibration: `table_threshold=0.30`, `dedup_iou=0.70` (both now defaults).
This is the gate run before recording Phase 2 layout detection as complete.

**Pass criteria (approximate):**
- `mean best_iou_crop` ≥ 0.88 on positive set
- FP crop rate ≤ 15% on negative set
- `failure` count = 0 or negligible

Run 6a (batch), then 6b (positive IoU), then 6c (false-positive).
Re-run the Step 4 manifest preview cell after 6a to see n=200 stats.

In [ ]:
# Step 6a - MVP batch (seed=42, n=200)
!python scripts/run_layout_batch.py --seed 42 --n 200 --require-table-gt --primary-threshold 0.3

In [ ]:
# Step 6b - positive IoU diagnostic (seed=42, n=200)
!python scripts/eval_layout_iou.py --seed 42 --n 200 --require-table-gt --primary-threshold 0.3

In [ ]:
# Step 6c - false-positive diagnostic (seed=42, n=200)
!python scripts/eval_layout_iou.py --seed 42 --n 200 --exclude-table-gt --primary-threshold 0.3

## Step 7 - End-to-end: Phase 2 crop → TATR structure recognition

Confirms the Phase 2 crops are compatible with the Phase 1 TATR structure model.
For each selected crop: runs inference → `normalize_tatr_prediction` → `validate_grid_geometry`.
Prints `rows`, `cols`, `cells`, `valid`, and failure reasons per crop. Writes `smoke_structure.csv`.

**Step 7 (baseline):** no band dedup — measures raw TATR output quality on DocLayNet crops.
**Step 7c (dedup):** `--dedup-bands` applies 1-D NMS to overlapping row/col bands before normalize.

`--n 50 --seed 42` samples 50 crops from the Step 6a batch (286 available).

In [ ]:
!python scripts/smoke_structure.py --n 50 --seed 42

In [ ]:
# Step 7b - smoke_structure.csv: inspect failure reasons for WARN crops
import pandas as pd
from src import config

df = pd.read_csv(config.LAYOUT_OUTPUT / "smoke_structure.csv")
print(f"OK: {df['valid'].sum()}  WARN: {(~df['valid']).sum()}  total: {len(df)}")
print()
warn = df[~df["valid"]].copy()
if len(warn):
    print("WARN crops:")
    print(warn[["crop", "rows", "cols", "cells", "failure_reasons"]].to_string(index=False))
    print()
    print("Failure reason counts:")
    all_reasons = [r.strip() for reasons in warn["failure_reasons"].dropna() for r in reasons.split(";") if r.strip()]
    from collections import Counter
    for reason, count in Counter(all_reasons).most_common():
        print(f"  {count:3d}  {reason}")
else:
    print("All crops valid.")

In [ ]:
# Step 7c - historical A/B reference only; dedup is now the default in normalize_tatr_prediction
# Before dedup: 37 OK / 13 WARN (Step 7b)
# After dedup:  50 OK /  0 WARN (this run, seed=42, n=50)
# !python scripts/smoke_structure.py --n 50 --seed 42 --dedup-bands  # flag no longer exists

In [ ]:
# Step 7d - full crop smoke: all 286 crops, dedup now default
# Result: 285 OK / 1 WARN (val_000670_table_1: rows=0 cols=4 -> no row boxes detected)
# WARN rate 0.35%, well under <=5% gate. Phase 2 crop -> structure handoff: PASSED.
!python scripts/smoke_structure.py --n 286 --seed 42